# Document Q&A Assistant with RAG

In this project, I will build a Retrieval-Augmented Generation (RAG) system that answers questions using information retrieved from a document set.

The system will split documents into chunks, generate embeddings, store them in a FAISS vector index, retrieve relevant chunks, and use an LLM to generate grounded answers with source citations.

I will also compare answers with and without retrieval and evaluate the retrieval stage separately.

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf

## Document Set

I use the official NumPy User Guide as the document set for the RAG system. The PDF is loaded, extracted page by page, and split into smaller chunks before embedding.

In [2]:
!wget -q https://numpy.org/doc/stable/numpy-user.pdf -O numpy_user_guide.pdf

In [3]:
from pypdf import PdfReader

reader = PdfReader("numpy_user_guide.pdf")

print("Number of pages:", len(reader.pages))

Number of pages: 721


## Text Extraction

I extract the text page by page and keep the page number as metadata. This metadata will later be used to show the source of each retrieved chunk.

In [4]:
documents = []

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text and text.strip():
        documents.append({
            "page": page_num,
            "text": text.strip()
        })

print("Pages with text:", len(documents))
print(documents[0])

Pages with text: 720
{'page': 1, 'text': 'NumPyUserGuide\nRelease 2.5.0\nWrittenbytheNumPycommunity\nJune28,2026'}


## Chunking

I split the extracted text into chunks of 800 characters with an overlap of 150 characters.

The overlap helps preserve information that may continue across chunk boundaries, while the chunk size keeps each retrieved context focused enough for similarity search.

In [5]:
chunk_size = 800
overlap = 150

chunks = []

for doc in documents:
    text = doc["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        chunks.append({
            "chunk_id": len(chunks),
            "page": doc["page"],
            "text": chunk_text
        })

        start += chunk_size - overlap

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 2363
{'chunk_id': 0, 'page': 1, 'text': 'NumPyUserGuide\nRelease 2.5.0\nWrittenbytheNumPycommunity\nJune28,2026'}


## Embeddings and FAISS Vector Store

I use the `all-MiniLM-L6-v2` Sentence-Transformers model to generate embeddings locally because it is free and lightweight. The embeddings are normalized and stored in a FAISS inner-product index, which acts as cosine similarity search for normalized vectors.

The FAISS index and chunk metadata are persisted to disk. If the saved files already exist, they are loaded instead of rebuilding the vector store.

In [6]:
import os
import faiss
import pickle
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

index_path = "numpy_rag.index"
chunks_path = "chunks.pkl"

if os.path.exists(index_path) and os.path.exists(chunks_path):

    index = faiss.read_index(index_path)

    with open(chunks_path, "rb") as f:
        chunks = pickle.load(f)

    print("Existing FAISS index loaded from disk.")

else:
    chunk_texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        chunk_texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings.astype("float32"))

    faiss.write_index(index, index_path)

    with open(chunks_path, "wb") as f:
        pickle.dump(chunks, f)

    print("New FAISS index created and saved to disk.")

print("Vectors in index:", index.ntotal)
print("Chunks:", len(chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Existing FAISS index loaded from disk.
Vectors in index: 2363
Chunks: 2363


## Retrieval Function

The retrieval function converts the user's question into an embedding and searches the FAISS index for the most similar document chunks.

The number of retrieved chunks is configurable using the `k` parameter.

In [7]:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "score": float(score)
        })

    return results

In [8]:
query = "What is broadcasting in NumPy?"

results = retrieve(query, k=3)

for result in results:
    print("Chunk:", result["chunk_id"])
    print("Page:", result["page"])
    print("Score:", round(result["score"], 3))
    print(result["text"][:500])
    print("-" * 80)

Chunk: 949
Page: 295
Score: 0.764
NumPyUserGuide,Release2.5.0
9.2.4 Broadcasting
See also:
Broadcasting
In Numeric, the ancestor of NumPy, broadcasting was implemented in several lines of code buried deep in
ufuncobject.c. In NumPy, the notion of broadcasting has been abstracted so that it can be performed in multiple
places. Broadcastingishandledbythefunction PyArray_Broadcast. Thisfunctionrequiresa PyArrayMulti-
IterObject(or something that is a binary equivalent) to be passed in. ThePyArrayMultiIterObjectkeeps
trackofthebroad
--------------------------------------------------------------------------------
Chunk: 285
Page: 100
Score: 0.7
NumPyUserGuide,Release2.5.0
4.5 Broadcasting
See also:
numpy.broadcast
The term broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations. Subject
tocertainconstraints,thesmallerarrayis“broadcast”acrossthelargerarraysothattheyhavecompatibleshapes. Broad-
castingprovidesameansofvectorizingarrayoperationssothatlo

## Local LLM with Ollama

I use `qwen2.5:1.5b` through Ollama as the local generation model. This keeps generation local and avoids a paid API.

In [9]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [10]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started.")

Ollama server started.


In [11]:
!ollama pull qwen2.5:1.5b

In [12]:
import requests

def generate_answer(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:1.5b",
            "prompt": prompt,
            "stream": False
        },
        timeout=300
    )
    response.raise_for_status()
    return response.json()["response"]

## Grounded Answer Generation

The RAG pipeline retrieves the top-k relevant chunks and places only those chunks in the LLM context.

The prompt explicitly separates the retrieved context from the user's question. The model is instructed to answer only from the retrieved context and to admit when the answer is not available in the documents.

In [13]:
def answer_with_rag(question, k=3):
    retrieved_chunks = retrieve(question, k=k)

    context = ""

    for chunk in retrieved_chunks:
        context += (
            f"[Chunk {chunk['chunk_id']}, Page {chunk['page']}]\n"
            f"{chunk['text']}\n\n"
        )

    prompt = f"""
You are a document question-answering assistant.

Answer the question ONLY using the CONTEXT below.

If the answer is not contained in the context, say:
"I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    answer = generate_answer(prompt)

    return answer, retrieved_chunks

In [14]:
question = "What is broadcasting in NumPy?"

answer, sources = answer_with_rag(question, k=3)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(
        f"Chunk {source['chunk_id']} | "
        f"Page {source['page']} | "
        f"Score: {source['score']:.3f}"
    )

ANSWER:
Broadcasting in NumPy is a feature that allows arrays with different shapes to be combined in a way that their elements are compatible for arithmetic operations. This is achieved by padding the smaller array with zeros to match the shape of the larger array along the appropriate dimensions, so that they can be treated as if they had the same shape. This mechanism is referred to as "broadcasting" and enables vectorized operations on arrays of different sizes without the need for explicit loops or unnecessary memory copies.

SOURCES:
Chunk 949 | Page 295 | Score: 0.764
Chunk 285 | Page 100 | Score: 0.700
Chunk 1664 | Page 513 | Score: 0.601


## Answer Without Retrieval

For comparison, I also ask the same LLM to answer without providing any retrieved document context.

This allows me to compare the model's parametric knowledge with the grounded RAG answer.

In [15]:
def answer_without_rag(question):
    prompt = f"""
Answer the following question using your own knowledge.

QUESTION:
{question}

ANSWER:
"""

    return generate_answer(prompt)

In [16]:
question = "What is broadcasting in NumPy?"

rag_answer, sources = answer_with_rag(question, k=3)
no_rag_answer = answer_without_rag(question)

print("WITH RETRIEVAL:")
print(rag_answer)

print("\nWITHOUT RETRIEVAL:")
print(no_rag_answer)

WITH RETRIEVAL:
Broadcasting in NumPy is a mechanism that allows arrays with different shapes to be used in arithmetic operations. It enables element-wise operations between arrays of different shapes by filling in the smaller array with appropriate values (usually zeros) so that the arrays can be combined. This process is known as "broadcasting" and is controlled by specific rules to ensure compatibility of shapes and size, while minimizing memory copies and enhancing efficiency.

WITHOUT RETRIEVAL:
Broadcasting is a feature in NumPy that allows arrays with different shapes to be combined and operated on element-wise. It automatically extends the smaller array (the "broadcasting target") to match the shape of the larger array (the "broadcasting source") by replicating the boundary values of the smaller array along its dimensions until the shapes of the two arrays match.

This concept is essential for performing operations that require broadcasting, such as element-wise multiplication,

## Grounding Proof

The same five questions are answered with retrieved document context and without retrieval. The side-by-side comparison shows whether retrieval changes the generated answers.

In [17]:
grounding_questions = [
    "According to the document, which function handles broadcasting in NumPy?",
    "According to the document, what object does PyArray_Broadcast require?",
    "According to the document, what does broadcasting allow to occur in C instead of Python?",
    "According to the document, what is a universal function in NumPy?",
    "According to the document, what is an array scalar in NumPy?"
]

In [18]:
import pandas as pd
comparison = []

for question in grounding_questions:
    rag_answer, sources = answer_with_rag(question, k=3)
    no_rag_answer = answer_without_rag(question)

    source_text = ", ".join(
        [f"Chunk {s['chunk_id']} (Page {s['page']})" for s in sources]
    )

    comparison.append({
        "Question": question,
        "With Retrieval": rag_answer,
        "Without Retrieval": no_rag_answer,
        "Sources": source_text
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

,Question,With Retrieval,Without Retrieval,Sources
0,"According to the document, which function hand...",The function that handles broadcasting in NumP...,"In NumPy, broadcasting is handled by the `np.n...","Chunk 949 (Page 295), Chunk 285 (Page 100), Ch..."
1,"According to the document, what object does Py...",PyArray_Broadcast requires a pointer to an ite...,PyArray_Broadcast requires an array or object ...,"Chunk 949 (Page 295), Chunk 950 (Page 295), Ch..."
2,"According to the document, what does broadcast...","According to the document, broadcasting allows...",The document mentions that broadcasting allows...,"Chunk 949 (Page 295), Chunk 285 (Page 100), Ch..."
3,"According to the document, what is a universal...",A universal function in NumPy is a callable ob...,"According to the information provided, a unive...","Chunk 2350 (Page 715), Chunk 960 (Page 297), C..."
4,"According to the document, what is an array sc...",An array scalar in NumPy is an instance of the...,An array scalar in NumPy is an element of an a...,"Chunk 1111 (Page 339), Chunk 81 (Page 34), Chu..."


## Out-of-Scope Handling

To test hallucination resistance, I ask questions about topics that are not covered by the NumPy document set.

The grounded assistant should decline to answer when the retrieved context does not contain the required information.

In [19]:
out_of_scope_questions = [
    "Who painted the Mona Lisa?",
    "What is the capital of Australia?"
]

for question in out_of_scope_questions:
    answer, sources = answer_with_rag(question, k=3)

    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)

    print("\n" + "-" * 80)

QUESTION:
Who painted the Mona Lisa?

ANSWER:
I don't know based on the provided documents.

--------------------------------------------------------------------------------
QUESTION:
What is the capital of Australia?

ANSWER:
I don't know based on the provided documents.

--------------------------------------------------------------------------------


## Retrieval Evaluation

I evaluate retrieval separately from answer generation using 10 labeled question-to-expected-chunk pairs.

For each question, retrieval is considered a hit if the expected chunk appears anywhere in the top 3 retrieved chunks. The final metric is Hit Rate@3.

In [20]:
evaluation_set = [
    ("What is broadcasting in NumPy?", 285),
    ("Which function handles broadcasting in NumPy?", 949),
    ("Where was broadcasting implemented in Numeric?", 949),
    ("What object does PyArray_Broadcast require?", 949),
    ("What does broadcasting allow to occur in C instead of Python?", 285),
    ("What is a universal function in NumPy?", 960),
    ("What are universal functions designed to do in NumPy?", 960),
    ("What is a scalar in NumPy?", 1111),
    ("What does the shape attribute of an ndarray represent?", 1124),
    ("What is an array scalar in NumPy?", 1111)
]

In [21]:
evaluation_results = []

for question, expected_chunk in evaluation_set:
    results = retrieve(question, k=3)

    retrieved_ids = [result["chunk_id"] for result in results]
    hit = expected_chunk in retrieved_ids

    evaluation_results.append({
        "Question": question,
        "Expected Chunk": expected_chunk,
        "Retrieved Top-3": retrieved_ids,
        "Hit": hit
    })

evaluation_df = pd.DataFrame(evaluation_results)

hit_rate = evaluation_df["Hit"].mean()

display(evaluation_df)

print(f"\nHit Rate@3: {hit_rate:.2%}")

,Question,Expected Chunk,Retrieved Top-3,Hit
0,What is broadcasting in NumPy?,285,"[949, 285, 1664]",True
1,Which function handles broadcasting in NumPy?,949,"[949, 285, 1664]",True
2,Where was broadcasting implemented in Numeric?,949,"[949, 285, 54]",True
3,What object does PyArray_Broadcast require?,949,"[949, 950, 951]",True
4,What does broadcasting allow to occur in C ins...,285,"[949, 285, 11]",True
5,What is a universal function in NumPy?,960,"[960, 2301, 1350]",True
6,What are universal functions designed to do in...,960,"[960, 2301, 12]",True
7,What is a scalar in NumPy?,1111,"[1111, 2320, 429]",True
8,What does the shape attribute of an ndarray re...,1124,"[1124, 1122, 141]",True
9,What is an array scalar in NumPy?,1111,"[1111, 81, 429]",True



Hit Rate@3: 100.00%


## Retrieval Gating Test

To verify that retrieval genuinely affects the generated answers, I compare the same questions with the vector store enabled and disabled.

When retrieval is disabled, no document context is supplied to the model. This demonstrates that the retrieved document chunks are actually used to ground the RAG answers.

In [22]:
def answer_with_retrieval_switch(question, retrieval_enabled=True, k=3):

    if retrieval_enabled:
        answer, sources = answer_with_rag(question, k=k)
        return answer, sources

    else:
        answer = answer_without_rag(question)
        return answer, []

In [23]:
gating_results = []

for question in grounding_questions:

    answer_on, sources = answer_with_retrieval_switch(
        question,
        retrieval_enabled=True
    )

    answer_off, _ = answer_with_retrieval_switch(
        question,
        retrieval_enabled=False
    )

    changed = answer_on.strip() != answer_off.strip()

    gating_results.append({
        "Question": question,
        "Retrieval ON": answer_on,
        "Retrieval OFF": answer_off,
        "Answer Changed": changed
    })

gating_df = pd.DataFrame(gating_results)

display(gating_df)

changed_count = gating_df["Answer Changed"].sum()

print(
    f"Answers changed when retrieval was disabled: "
    f"{changed_count}/{len(gating_df)}"
)

,Question,Retrieval ON,Retrieval OFF,Answer Changed
0,"According to the document, which function hand...",The function that handles broadcasting in NumP...,The function that handles broadcasting in NumP...,True
1,"According to the document, what object does Py...",PyArray_Broadcast requires a PyArrayMultiIterO...,PyArray_Broadcast requires a tuple of object t...,True
2,"According to the document, what does broadcast...",Broadcasting allows vectorized array operation...,"According to the document, broadcasting allows...",True
3,"According to the document, what is a universal...",A universal function in NumPy is a callable ob...,A universal function in NumPy is a function th...,True
4,"According to the document, what is an array sc...",An array scalar in NumPy is an instance of the...,"In NumPy, an array scalar is an element of a N...",True


Answers changed when retrieval was disabled: 5/5


## Results

The RAG system successfully retrieves relevant chunks from the NumPy User Guide and uses only the retrieved context to generate grounded answers.

- The document was split into 2363 chunks using a chunk size of 800 characters and an overlap of 150 characters.
- Embeddings were generated using `all-MiniLM-L6-v2`.
- The embeddings were indexed with FAISS and the index was persisted to disk.
- Retrieval uses configurable top-k similarity search.
- Retrieval evaluation achieved a Hit Rate@3 of 100% on 10 labeled questions.
- The grounded assistant correctly declined to answer both out-of-scope questions.
- Answers with retrieval differed from answers without retrieval in the grounding comparison.
- Disabling retrieval changed the answers on all 5 grounding-test questions, demonstrating that retrieval genuinely affects generation.
- Every grounded answer reports the retrieved chunk IDs and source page numbers.

## Conclusion

This project demonstrates a complete Retrieval-Augmented Generation pipeline using Sentence-Transformers, FAISS, and a local LLM through Ollama. The retrieval and generation stages were evaluated separately. The results show that the system can retrieve relevant document chunks, generate answers grounded in those chunks, cite its sources, and avoid answering questions that are outside the document set. The retrieval ON/OFF comparison also demonstrates that the vector store genuinely influences the generated answers.